# Ephesos examples with exomoons and circumplanetary disks

These examples use deterministic forward models. The moon's circular orbit is derived from the specified planet and moon masses and their separation. The disk model uses an opaque planet with a uniform, partially transmitting annulus. Neither example is a fit to observed data.

## 1. Environment Setup and Package Installation

Run this notebook from `examples/exomoon`. If the package or pandas is unavailable, install the local package and plotting dependency with `%pip install -e ../.. pandas`.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import os
from pathlib import Path
import sys

REPOSITORY_PATH = Path.cwd().resolve().parents[1]
if str(REPOSITORY_PATH) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_PATH))
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
for dependency_repository in ("tdpy", "chalcedon"):
    dependency_path = REPOSITORY_PATH.parent / dependency_repository
    if dependency_path.is_dir() and str(dependency_path) not in sys.path:
        sys.path.insert(0, str(dependency_path))
os.environ.setdefault("EPHESOS_PATH", str(REPOSITORY_PATH))

from tdpy.verbosity import print

for package_name in ("ephesos", "numpy", "scipy", "matplotlib", "pandas"):
    try:
        print(f"{package_name}: {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}: not installed; run %pip install -e ../.. pandas")

import numpy as np
np.random.seed(0)

## 2. Import Ephesos and Scientific Python Dependencies

Set consistent figure styling and a compact display helper for transit metrics.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from astropy.constants import G, M_earth, R_sun
from IPython.display import display

import ephesos

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "font.size": 10,
})


def display_metrics(metrics):
    """Display a metric table with compact numerical formatting."""
    return metrics.style.format("{:.4g}")

## 3. Define Stellar, Planet, and Orbit Configuration

The model inputs use radius ratios and dimensionless orbital geometry. The moon period is derived from Kepler's law, including both companion masses. The bounds below are example-domain checks, not population priors.

In [ ]:
STAR_PARAMETERS = {
    "mass_solar": 1.0,  # [M_sun]
    "radius_solar": 1.0,  # [R_sun]
    "limb_darkening_coefficients": (0.4, 0.25),
}
PLANET_PARAMETERS = {
    "mass_earth": 10.0,  # [M_earth]
    "radius_ratio": 0.09,
}
MOON_PARAMETERS = {
    "mass_earth": 0.1,  # [M_earth]
    "radius_ratio": 0.025,
    "separation_planet_radii": 5.0,
}
ORBIT_PARAMETERS = {
    "planet_period_days": 4.0,  # [day]
    "summed_radius_to_semimajor_axis": 0.1,
    "cosine_inclination": 0.015,
    "transit_epoch_days": 0.0,  # [day]
}
PARAMETER_BOUNDS = {
    "mass_positive": (0.0, np.inf),
    "radius_ratio_positive": (0.0, 1.0),
    "separation_planet_radii": (1.0, 20.0),
}

planet_radius_m = (
    PLANET_PARAMETERS["radius_ratio"] * STAR_PARAMETERS["radius_solar"] * R_sun.value
)  # [m]
moon_separation_m = MOON_PARAMETERS["separation_planet_radii"] * planet_radius_m  # [m]
moon_system_mass_kg = (
    PLANET_PARAMETERS["mass_earth"] + MOON_PARAMETERS["mass_earth"]
) * M_earth.value  # [kg]
MOON_PARAMETERS["period_days"] = float(
    2.0 * np.pi * np.sqrt(moon_separation_m**3 / (G.value * moon_system_mass_kg)) / 86400.0
)  # [day]

assert PLANET_PARAMETERS["mass_earth"] > 0.0
assert MOON_PARAMETERS["mass_earth"] > 0.0
assert PARAMETER_BOUNDS["separation_planet_radii"][0] <= MOON_PARAMETERS["separation_planet_radii"] <= PARAMETER_BOUNDS["separation_planet_radii"][1]
MOON_PARAMETERS

## 4. Build a Baseline Planet-Only Transit Model

Compare the established Ephesos transit wrapper with the planet-only limit of the planet-moon image-grid model. Small differences reflect their distinct numerical integrations.

In [ ]:
TIME_HOURS = np.linspace(-6.0, 6.0, 241)  # [hour]
TIME_DAYS = TIME_HOURS / 24.0  # [day]
BASELINE_FLUX = ephesos.evaluate_transit_model(
    TIME_DAYS,
    period_days=ORBIT_PARAMETERS["planet_period_days"],
    radius_ratio=PLANET_PARAMETERS["radius_ratio"],
    summed_radius_to_semimajor_axis=ORBIT_PARAMETERS["summed_radius_to_semimajor_axis"],
    cosine_inclination=ORBIT_PARAMETERS["cosine_inclination"],
    limb_darkening_coefficients=STAR_PARAMETERS["limb_darkening_coefficients"],
)
PLANET_ONLY_FLUX = ephesos.evaluate_exomoon_transit_model(
    TIME_DAYS,
    planet_period_days=ORBIT_PARAMETERS["planet_period_days"],
    planet_radius_ratio=PLANET_PARAMETERS["radius_ratio"],
    summed_radius_to_semimajor_axis=ORBIT_PARAMETERS["summed_radius_to_semimajor_axis"],
    moon_radius_ratio=0.0,
    moon_separation_planet_radii=MOON_PARAMETERS["separation_planet_radii"],
    moon_period_days=MOON_PARAMETERS["period_days"],
    cosine_inclination=ORBIT_PARAMETERS["cosine_inclination"],
    limb_darkening_coefficients=STAR_PARAMETERS["limb_darkening_coefficients"],
    grid_size=201,
)

figure, axis = plt.subplots(figsize=(7.2, 3.6))
axis.plot(TIME_HOURS, BASELINE_FLUX, color="black", label="Transit wrapper")
axis.plot(TIME_HOURS, PLANET_ONLY_FLUX, color="#16697A", linestyle="--", label="Image-grid planet only")
axis.set(xlabel="Time from transit [hour]", ylabel="Relative flux", title="Planet-only reference transit")
axis.grid(False)
axis.legend(fancybox=True, framealpha=1.0)
figure.tight_layout()
plt.show()
print(f"Transit depth: {(1.0 - PLANET_ONLY_FLUX.min()) * 100.0:.4g}%")

## 5. Add an Exomoon and Generate Moon-Planet Transit Signals

Use the same planet orbit for leading and trailing moon phases. A moon transit appears as an additional flux deficit whose timing shifts with orbital phase.

In [ ]:
MOON_PHASES = {
    "Leading moon": np.pi / 2.0,  # [rad]
    "Trailing moon": -np.pi / 2.0,  # [rad]
}
EXOMOON_FLUXES = {
    label: ephesos.evaluate_exomoon_transit_model(
        TIME_DAYS,
        planet_period_days=ORBIT_PARAMETERS["planet_period_days"],
        planet_radius_ratio=PLANET_PARAMETERS["radius_ratio"],
        summed_radius_to_semimajor_axis=ORBIT_PARAMETERS["summed_radius_to_semimajor_axis"],
        moon_radius_ratio=MOON_PARAMETERS["radius_ratio"],
        moon_separation_planet_radii=MOON_PARAMETERS["separation_planet_radii"],
        moon_period_days=MOON_PARAMETERS["period_days"],
        moon_orbital_phase_radians=phase,
        cosine_inclination=ORBIT_PARAMETERS["cosine_inclination"],
        limb_darkening_coefficients=STAR_PARAMETERS["limb_darkening_coefficients"],
        grid_size=201,
    )
    for label, phase in MOON_PHASES.items()
}

figure, (flux_axis, residual_axis) = plt.subplots(
    2, 1, figsize=(7.2, 5.5), sharex=True, gridspec_kw={"height_ratios": (2, 1)}
)
flux_axis.plot(TIME_HOURS, PLANET_ONLY_FLUX, color="black", label="Planet only")
for label, flux in EXOMOON_FLUXES.items():
    color = "#16697A" if label == "Leading moon" else "#C73E1D"
    flux_axis.plot(TIME_HOURS, flux, color=color, label=label)
    residual_axis.plot(TIME_HOURS, 1e3 * (flux - PLANET_ONLY_FLUX), color=color, label=label)
flux_axis.set(ylabel="Relative flux", title="Transit signals for two moon phases")
residual_axis.set(xlabel="Time from planetary transit [hour]", ylabel="Moon signal [ppt]")
for axis in (flux_axis, residual_axis):
    axis.grid(False)
flux_axis.legend(fancybox=True, framealpha=1.0)
residual_axis.axhline(0.0, color="black", linewidth=1.0)
figure.tight_layout()
plt.show()

## 6. Add a Circumplanetary Disk Model

The disk is a uniform annulus with inner and outer edges in planet radii. Its normal optical depth becomes line-of-sight optical depth after accounting for the disk inclination. The planet itself remains opaque.

In [ ]:
DISK_PARAMETERS = {
    "inner_radius_planet_radii": 1.5,
    "outer_radius_planet_radii": 4.0,
    "inclination_degrees": 45.0,  # [degree]
    "position_angle_degrees": 0.0,  # [degree]
    "optical_depth": 0.8,
}


def evaluate_disk_curve(inclination_degrees, optical_depth):
    """Run the public Ephesos planet-with-disk forward model."""
    return ephesos.evaluate_planet_with_disk_model(
        TIME_DAYS,
        period_days=ORBIT_PARAMETERS["planet_period_days"],
        planet_radius_ratio=PLANET_PARAMETERS["radius_ratio"],
        summed_radius_to_semimajor_axis=ORBIT_PARAMETERS["summed_radius_to_semimajor_axis"],
        disk_inner_radius_planet_radii=DISK_PARAMETERS["inner_radius_planet_radii"],
        disk_outer_radius_planet_radii=DISK_PARAMETERS["outer_radius_planet_radii"],
        disk_inclination_degrees=inclination_degrees,
        disk_position_angle_degrees=DISK_PARAMETERS["position_angle_degrees"],
        disk_optical_depth=optical_depth,
        cosine_inclination=ORBIT_PARAMETERS["cosine_inclination"],
        limb_darkening_coefficients=STAR_PARAMETERS["limb_darkening_coefficients"],
        grid_size=201,
    )


DISK_FLUX = evaluate_disk_curve(
    DISK_PARAMETERS["inclination_degrees"], DISK_PARAMETERS["optical_depth"]
)
TRANSPARENT_DISK_FLUX = evaluate_disk_curve(DISK_PARAMETERS["inclination_degrees"], 0.0)
figure, axis = plt.subplots(figsize=(7.2, 3.8))
axis.plot(TIME_HOURS, PLANET_ONLY_FLUX, color="black", label="Planet only")
axis.plot(TIME_HOURS, TRANSPARENT_DISK_FLUX, color="#16697A", linestyle="--", label="Transparent disk")
axis.plot(TIME_HOURS, DISK_FLUX, color="#C73E1D", label="Disk, tau = 0.8")
axis.set(xlabel="Time from transit [hour]", ylabel="Relative flux", title="Circumplanetary disk transit")
axis.grid(False)
axis.legend(fancybox=True, framealpha=1.0)
figure.tight_layout()
plt.show()

## 7 Compare Planet-Only, Exomoon, and Disk Light Curves

Report peak-depth changes, root-mean-square residuals, and transit-center offsets relative to the grid-integrated planet-only curve.

In [ ]:
ALL_SCENARIOS = {"Planet only": PLANET_ONLY_FLUX, **EXOMOON_FLUXES, "Circumplanetary disk": DISK_FLUX}
SCENARIO_BASELINES = {
    "Planet only": PLANET_ONLY_FLUX,
    "Leading moon": PLANET_ONLY_FLUX,
    "Trailing moon": PLANET_ONLY_FLUX,
    "Circumplanetary disk": TRANSPARENT_DISK_FLUX,
}


def transit_metrics(relative_flux, baseline_flux):
    """Summarize depth, RMS residual, and minimum-flux timing versus baseline."""
    residual = relative_flux - baseline_flux
    return {
        "depth_change_ppt": 1e3 * (baseline_flux.min() - relative_flux.min()),
        "rms_residual_ppt": 1e3 * np.sqrt(np.mean(residual**2)),
        "center_offset_hour": TIME_HOURS[np.argmin(relative_flux)] - TIME_HOURS[np.argmin(baseline_flux)],
    }


COMPARISON_METRICS = pd.DataFrame(
    {
        label: transit_metrics(flux, SCENARIO_BASELINES[label])
        for label, flux in ALL_SCENARIOS.items()
    }
).T

display_metrics(COMPARISON_METRICS)
figure, (flux_axis, residual_axis) = plt.subplots(
    2, 1, figsize=(7.2, 6.0), sharex=True, gridspec_kw={"height_ratios": (2, 1)}
)
for label, flux in ALL_SCENARIOS.items():
    baseline_flux = SCENARIO_BASELINES[label]
    flux_axis.plot(TIME_HOURS, flux, label=label)
    residual_axis.plot(TIME_HOURS, 1e3 * (flux - baseline_flux), label=label)
flux_axis.set(ylabel="Relative flux", title="Planet, moon, and disk comparisons")
residual_axis.set(xlabel="Time from transit [hour]", ylabel="Residual [ppt]")
for axis in (flux_axis, residual_axis):
    axis.grid(False)
flux_axis.legend(fancybox=True, framealpha=1.0)
residual_axis.axhline(0.0, color="black", linewidth=1.0)
figure.tight_layout()
plt.show()

## 8. Run Parameter Sweeps and Visual Diagnostics

Vary the moon's orbital distance with its Keplerian period recomputed at each step. Map the disk signal over normal optical depth and inclination using the Ephesos forward model.

In [ ]:
MOON_SEPARATION_SWEEP = np.linspace(2.0, 8.0, 7)
MOON_SWEEP_SIGNAL_PPT = []
for separation in MOON_SEPARATION_SWEEP:
    period = MOON_PARAMETERS["period_days"] * (
        separation / MOON_PARAMETERS["separation_planet_radii"]
    ) ** 1.5  # [day]
    flux = ephesos.evaluate_exomoon_transit_model(
        TIME_DAYS,
        planet_period_days=ORBIT_PARAMETERS["planet_period_days"],
        planet_radius_ratio=PLANET_PARAMETERS["radius_ratio"],
        summed_radius_to_semimajor_axis=ORBIT_PARAMETERS["summed_radius_to_semimajor_axis"],
        moon_radius_ratio=MOON_PARAMETERS["radius_ratio"],
        moon_separation_planet_radii=separation,
        moon_period_days=period,
        moon_orbital_phase_radians=np.pi / 2.0,
        cosine_inclination=ORBIT_PARAMETERS["cosine_inclination"],
        limb_darkening_coefficients=STAR_PARAMETERS["limb_darkening_coefficients"],
        grid_size=151,
    )
    MOON_SWEEP_SIGNAL_PPT.append(1e3 * np.max(np.abs(flux - PLANET_ONLY_FLUX)))

DISK_OPTICAL_DEPTHS = np.linspace(0.0, 2.0, 7)
DISK_INCLINATIONS_DEGREES = np.linspace(0.0, 80.0, 5)  # [degree]
DISK_SWEEP_SIGNAL_PPT = np.empty((DISK_INCLINATIONS_DEGREES.size, DISK_OPTICAL_DEPTHS.size))
for inclination_index, inclination in enumerate(DISK_INCLINATIONS_DEGREES):
    matched_baseline = evaluate_disk_curve(inclination, 0.0)
    for depth_index, optical_depth in enumerate(DISK_OPTICAL_DEPTHS):
        flux = evaluate_disk_curve(inclination, optical_depth)
        DISK_SWEEP_SIGNAL_PPT[inclination_index, depth_index] = 1e3 * np.max(
            np.abs(flux - matched_baseline)
        )

figure, (moon_axis, disk_axis) = plt.subplots(1, 2, figsize=(10.0, 4.0))
moon_axis.plot(MOON_SEPARATION_SWEEP, MOON_SWEEP_SIGNAL_PPT, marker="o", color="#16697A")
moon_axis.set(xlabel="Moon separation [planet radii]", ylabel="Maximum departure [ppt]", title="Moon separation sweep")
image = disk_axis.imshow(
    DISK_SWEEP_SIGNAL_PPT,
    origin="lower",
    aspect="auto",
    extent=(DISK_OPTICAL_DEPTHS.min(), DISK_OPTICAL_DEPTHS.max(), DISK_INCLINATIONS_DEGREES.min(), DISK_INCLINATIONS_DEGREES.max()),
    cmap="viridis",
)
disk_axis.set(xlabel="Normal optical depth", ylabel="Disk inclination [degree]", title="Disk signal sweep")
figure.colorbar(image, ax=disk_axis, label="Maximum departure [ppt]")
for axis in (moon_axis, disk_axis):
    axis.grid(False)
figure.tight_layout()
plt.show()

## 9. Create Reusable Simulation Utilities

Keep the parameterized model calls and metric calculation reusable, verify limiting cases, and save figures under the example's `visuals/` directory and comparison tables under the repository's ignored `data/` directory.

In [ ]:
from tdpy.verbosity import print


def evaluate_moon_curve(separation_planet_radii, orbital_phase_radians):
    """Evaluate a moon signal while preserving its mass-consistent period."""
    period_days = MOON_PARAMETERS["period_days"] * (
        separation_planet_radii / MOON_PARAMETERS["separation_planet_radii"]
    ) ** 1.5  # [day]
    return ephesos.evaluate_exomoon_transit_model(
        TIME_DAYS,
        planet_period_days=ORBIT_PARAMETERS["planet_period_days"],
        planet_radius_ratio=PLANET_PARAMETERS["radius_ratio"],
        summed_radius_to_semimajor_axis=ORBIT_PARAMETERS["summed_radius_to_semimajor_axis"],
        moon_radius_ratio=MOON_PARAMETERS["radius_ratio"],
        moon_separation_planet_radii=separation_planet_radii,
        moon_period_days=period_days,
        moon_orbital_phase_radians=orbital_phase_radians,
        cosine_inclination=ORBIT_PARAMETERS["cosine_inclination"],
        limb_darkening_coefficients=STAR_PARAMETERS["limb_darkening_coefficients"],
        grid_size=201,
    )


np.testing.assert_allclose(
    ephesos.evaluate_exomoon_transit_model(
        TIME_DAYS,
        planet_period_days=ORBIT_PARAMETERS["planet_period_days"],
        planet_radius_ratio=PLANET_PARAMETERS["radius_ratio"],
        summed_radius_to_semimajor_axis=ORBIT_PARAMETERS["summed_radius_to_semimajor_axis"],
        moon_radius_ratio=0.0,
        moon_separation_planet_radii=MOON_PARAMETERS["separation_planet_radii"],
        moon_period_days=MOON_PARAMETERS["period_days"],
        cosine_inclination=ORBIT_PARAMETERS["cosine_inclination"],
        grid_size=201,
    ),
    PLANET_ONLY_FLUX,
)
np.testing.assert_allclose(
    evaluate_disk_curve(0.0, 0.0),
    evaluate_disk_curve(80.0, 0.0),
    atol=1e-12,
)
assert all(np.isfinite(flux).all() for flux in ALL_SCENARIOS.values())

MOON_SWEEP_TABLE = pd.DataFrame({
    "separation_planet_radii": MOON_SEPARATION_SWEEP,
    "maximum_departure_ppt": MOON_SWEEP_SIGNAL_PPT,
})
DISK_SWEEP_TABLE = pd.DataFrame(
    DISK_SWEEP_SIGNAL_PPT,
    index=pd.Index(DISK_INCLINATIONS_DEGREES, name="inclination_degrees"),
    columns=pd.Index(DISK_OPTICAL_DEPTHS, name="normal_optical_depth"),
)

visuals_path = Path("visuals")
visuals_path.mkdir(parents=True, exist_ok=True)
data_path = ephesos.get_data_path()
data_path.mkdir(parents=True, exist_ok=True)
figure, (moon_axis, disk_axis) = plt.subplots(1, 2, figsize=(10.0, 4.0))
moon_axis.plot(MOON_SWEEP_TABLE["separation_planet_radii"], MOON_SWEEP_TABLE["maximum_departure_ppt"], marker="o", color="#16697A")
moon_axis.set(xlabel="Moon separation [planet radii]", ylabel="Maximum departure [ppt]", title="Moon separation")
image = disk_axis.imshow(
    DISK_SWEEP_TABLE.to_numpy(),
    origin="lower",
    aspect="auto",
    extent=(DISK_OPTICAL_DEPTHS.min(), DISK_OPTICAL_DEPTHS.max(), DISK_INCLINATIONS_DEGREES.min(), DISK_INCLINATIONS_DEGREES.max()),
    cmap="viridis",
)
disk_axis.set(xlabel="Normal optical depth", ylabel="Disk inclination [degree]", title="Disk signal")
figure.colorbar(image, ax=disk_axis, label="Maximum departure [ppt]")
for axis in (moon_axis, disk_axis):
    axis.grid(False)
figure.tight_layout()
figure_path = visuals_path / "exomoon_disk_parameter_sweep.png"
print(f"Writing to {figure_path}...")
figure.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.close(figure)
metrics_path = data_path / "exomoon_disk_metrics.csv"
print(f"Writing to {metrics_path}...")
COMPARISON_METRICS.to_csv(metrics_path)
display_metrics(COMPARISON_METRICS)
display(MOON_SWEEP_TABLE)
display(DISK_SWEEP_TABLE)

## 10. Explore the Sweeps Interactively

The companion Bokeh view makes the moon-separation curve and disk-parameter grid zoomable and exposes each plotted value on hover. The static Matplotlib figure above remains available for reports. Install `python -m pip install -e '.[bokeh]'` before running this cell.

In [ ]:
from bokeh.layouts import row
from bokeh.models import ColorBar, ColumnDataSource, HoverTool, LinearColorMapper
from bokeh.palettes import Viridis256
from bokeh.plotting import figure
from tdpy.plotting import save_bokeh_html

moon_source = ColumnDataSource({
    "separation": MOON_SEPARATION_SWEEP,
    "signal": MOON_SWEEP_SIGNAL_PPT,
})
moon_plot = figure(
    title="Moon signal versus orbital separation",
    width=520,
    height=390,
    sizing_mode="stretch_width",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
moon_plot.line("separation", "signal", source=moon_source, color="#16697A", line_width=2.0)
moon_plot.scatter("separation", "signal", source=moon_source, color="#16697A", size=8)
moon_plot.add_tools(HoverTool(tooltips=[("Separation [planet radii]", "@separation{0.0}"), ("Maximum signal [ppt]", "@signal{0.000}")]))
moon_plot.xaxis.axis_label = "Moon separation [planet radii]"
moon_plot.yaxis.axis_label = "Maximum transit departure [ppt]"
moon_plot.grid.visible = False

optical_depth_grid = np.tile(DISK_OPTICAL_DEPTHS, DISK_INCLINATIONS_DEGREES.size)
inclination_grid = np.repeat(DISK_INCLINATIONS_DEGREES, DISK_OPTICAL_DEPTHS.size)
disk_source = ColumnDataSource({
    "optical_depth": optical_depth_grid,
    "inclination": inclination_grid,
    "signal": DISK_SWEEP_SIGNAL_PPT.ravel(),
})
color_mapper = LinearColorMapper(
    palette=Viridis256,
    low=float(np.nanmin(DISK_SWEEP_SIGNAL_PPT)),
    high=float(np.nanmax(DISK_SWEEP_SIGNAL_PPT)),
)
disk_plot = figure(
    title="Disk signal across inclination and optical depth",
    width=560,
    height=390,
    sizing_mode="stretch_width",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
disk_plot.rect(
    x="optical_depth",
    y="inclination",
    width=float(np.min(np.diff(DISK_OPTICAL_DEPTHS))),
    height=float(np.min(np.diff(DISK_INCLINATIONS_DEGREES))),
    source=disk_source,
    fill_color={"field": "signal", "transform": color_mapper},
    line_color="white",
)
disk_plot.add_layout(ColorBar(color_mapper=color_mapper, title="Maximum departure [ppt]"), "right")
disk_plot.add_tools(HoverTool(tooltips=[("Optical depth", "@optical_depth{0.00}"), ("Inclination [degree]", "@inclination{0}"), ("Maximum signal [ppt]", "@signal{0.000}")]))
disk_plot.xaxis.axis_label = "Normal optical depth"
disk_plot.yaxis.axis_label = "Disk inclination [degree]"
disk_plot.grid.visible = False

bokeh_path = visuals_path / "exomoon_disk_parameter_sweep.html"
save_bokeh_html(row(moon_plot, disk_plot, sizing_mode="stretch_width"), bokeh_path, title="Exomoon and disk parameter sweeps")